# Part A

In [72]:
import torch
import torch.nn as nn

torch.manual_seed(42)

names = ["Aridia", 
         "Borland", 
         "Cyrenia",
         "Deltora",
         "Estovia",
         "Freland",
         "Galenia",
         "Hesperia"
        ]

A_full = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 1, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 1], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [1, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 1, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

degree = A_full.sum(dim=1)
degree_ave = torch.mean(degree)
unique_edges = torch.sum(degree)/2

X = torch.tensor([
    [5.8, 3.1, 4.2, 46, 6.2],
    [5.1, 3.8, 5.0, 42, 5.7],
    [6.4, 2.7, 3.8, 51, 7.1],
    [3.2, 6.5, 7.4, 30, 3.1],
    [2.8, 7.2, 8.1, 27, 2.8],
    [4.4, 4.1, 5.6, 39, 4.9],
    [5.6, 3.0, 4.5, 48, 6.5],
    [3.5, 5.8, 6.9, 33, 3.6]
    ], dtype=torch.float32
)

X_mean = X.mean(dim=0)
X_std = X.std(dim=0)
X_scaled = (X - X_mean)/X_std

print(f"Diagonal: {torch.diag(A_full).numpy()}")
print(f"Degree of Country: {degree.numpy()}")
print(f"Mean Degree: {degree_ave.numpy()}")
print(f"Unique Edges: {unique_edges.numpy()}")

Diagonal: [0. 0. 0. 0. 0. 0. 0. 0.]
Degree of Country: [3. 3. 3. 3. 2. 5. 4. 3.]
Mean Degree: 3.25
Unique Edges: 13.0


Summing all entries of a symmetric adjacency matrix double-counts undirected edges because the connection is counted on both sides. For example, A--B, B is counted on row A and A is counted on row B which essentially double counts the same connection.

Standardizing the feature matrix is important because each feature can have different scales so larger data in a feature will give that feature a higher weight and standardizing will remove that problem making each feature equally important.

# Part B

In [73]:
# B1
A_train = torch.tensor([
    #A  B  C  D  E  F  G  H
    [0, 1, 1, 0, 0, 0, 0, 0], # A
    [1, 0, 0, 0, 0, 1, 1, 0], # B
    [1, 0, 0, 0, 0, 1, 1, 0], # C
    [0, 0, 0, 0, 1, 1, 0, 0], # D
    [0, 0, 0, 1, 0, 0, 0, 1], # E
    [0, 1, 1, 1, 0, 0, 1, 1], # F
    [0, 1, 1, 0, 0, 1, 0, 0], # G
    [0, 0, 0, 0, 1, 1, 0, 0] # H
    ], dtype=torch.float32
)

train_degree = A_train.sum(dim=1)
train_unique_edges = torch.sum(train_degree)/2
print(f"Unique Edges: {train_unique_edges.numpy()}")
print(f"Training graph contains 11 unique edges: {train_unique_edges.numpy() == 11.0}")


Unique Edges: 11.0
Training graph contains 11 unique edges: True


In [74]:
# B2
def aggregate(A, H):
    I = torch.eye(A.shape[0])
    A_hat = A + I
    degree_hat = A_hat.sum(dim=1)
    degree_hat = degree_hat.clamp(min=1) # handle degrees safely, basically does max(degree_hat, 1)
    D_inverse = torch.diag(1.0 / degree_hat)
    X_aggregated = D_inverse @ A_hat @ H
    assert X_aggregated.shape == H.shape, "X_aggregated's shape == H.shape"
    # print(f"X_aggregated has the same shape as H: {X_aggregated.shape == H.shape}")
    return X_aggregated

In [83]:
# B3
class GNNEncoder(nn.Module):
    def __init__(self, input_features=5, hidden_size=8, embedding_size=3):
        super().__init__()
        self.linear_one = nn.Linear(input_features, hidden_size)
        self.activation = nn.ReLU()
        self.linear_two = nn.Linear(hidden_size, embedding_size)

    def forward(self, A, X):
        layer_one = self.linear_one(aggregate(A, X))
        X_activated = self.activation(layer_one)
        layer_two = self.linear_two(aggregate(A, X_activated))
        return layer_two

class GraphAutoencoder(nn.Module):
    def __init__(self, input_features_size, hidden_size, embedding_size):
        super().__init__()
        self.encoder = GNNEncoder(input_features_size, hidden_size, embedding_size)

    def forward(self, A, X):
        Z = self.encoder(A, X)
        scores = Z @ Z.T
        A_reconstructed = torch.sigmoid(scores)
        return A_reconstructed, Z

# Reporing helpers
def unique_pairs(A):
    # Undirected edges occur twice in A. Score each pair once, excluding the
    # diagonal: self-loops support aggregation but are not prediction targets.
    return torch.triu_indices(
        A.shape[0], A.shape[1], offset=1,
        device=A.device,
    )

def report(model, A, X):
    model.eval()
    with torch.no_grad():
        probabilities, embeddings = model(A, X)
        rows, cols = unique_pairs(A)
        loss = nn.functional.binary_cross_entropy(
            probabilities[rows, cols], A[rows, cols]
        ).item()
        accuracy = (
            (probabilities[rows, cols] >= 0.5) == A[rows, cols].bool()
        ).float().mean().item()

    print("Supplied adjacency matrix:")
    print(A)
    print("Reconstructed edge probabilities (diagonal NOT forced to zero):")
    print(probabilities)

    print("Reconstructed adjacency at threshold 0.5:")
    print((probabilities >= 0.5).int())
    print(f"Pairwise BCE: {loss:.4f}; pairwise accuracy: {accuracy:.0%}")
    print("Edge                 Target    Score")
    for i, j in zip(rows.tolist(), cols.tolist()):
        print(f"{names[i] + '--' + names[j]:20s}"
            f" {int(A[i, j]):3d}      {probabilities[i, j]:.4f}")

    print("\nCountry: Embedding")
    for i, embedding in enumerate(embeddings):
        print(f"{names[i]}: {embedding}") 

    return probabilities, embeddings
              
    

In [85]:
# B4
torch.manual_seed(42)
INPUT_FEATURES = X_scaled
ENCODER_GRAPH = A_train
BASELINE_RECONSTRUCTION_TARGET = A_train
HIDDEN_DIMENSION = 8
EMBEDDING_DIMENSION = 3
LEARNING_RATE = 0.01
EPOCHS = 2000

model = GraphAutoencoder(INPUT_FEATURES.shape[1], HIDDEN_DIMENSION, EMBEDDING_DIMENSION)
loss_function = nn.BCELoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=LEARNING_RATE
)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    A_reconstructed, Z = model(ENCODER_GRAPH, INPUT_FEATURES)
    loss = loss_function(A_reconstructed, BASELINE_RECONSTRUCTION_TARGET)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch}: loss = {loss}")
    
    if epoch == EPOCHS - 1:
        print(f"Final Loss: {loss}\n")
        
report(model, ENCODER_GRAPH, INPUT_FEATURES)

Epoch 0: loss = 0.7324681282043457
Epoch 200: loss = 0.599016547203064
Epoch 400: loss = 0.5987579226493835
Epoch 600: loss = 0.5982092618942261
Epoch 800: loss = 0.5979307293891907
Epoch 1000: loss = 0.5976467728614807
Epoch 1200: loss = 0.5973795652389526
Epoch 1400: loss = 0.5971369743347168
Epoch 1600: loss = 0.5967096090316772
Epoch 1800: loss = 0.5961896181106567
Final Loss: 0.5949258804321289

Supplied adjacency matrix:
tensor([[0., 1., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 1.],
        [0., 1., 1., 1., 0., 0., 1., 1.],
        [0., 1., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 1., 0., 0.]])
Reconstructed edge probabilities (diagonal NOT forced to zero):
tensor([[0.7125, 0.7048, 0.7126, 0.2945, 0.2021, 0.5682, 0.7180, 0.3176],
        [0.7048, 0.7193, 0.7195, 0.2923, 0.1910, 0.5901, 0.7418, 0.3232],
        [0.7126, 0.7

(tensor([[0.7125, 0.7048, 0.7126, 0.2945, 0.2021, 0.5682, 0.7180, 0.3176],
         [0.7048, 0.7193, 0.7195, 0.2923, 0.1910, 0.5901, 0.7418, 0.3232],
         [0.7126, 0.7195, 0.7223, 0.2880, 0.1887, 0.5847, 0.7391, 0.3166],
         [0.2945, 0.2923, 0.2880, 0.7030, 0.7986, 0.4234, 0.2750, 0.6767],
         [0.2021, 0.1910, 0.1887, 0.7986, 0.9028, 0.3691, 0.1674, 0.7621],
         [0.5682, 0.5901, 0.5847, 0.4234, 0.3691, 0.5446, 0.6067, 0.4410],
         [0.7180, 0.7418, 0.7391, 0.2750, 0.1674, 0.6067, 0.7685, 0.3107],
         [0.3176, 0.3232, 0.3166, 0.6767, 0.7621, 0.4410, 0.3107, 0.6558]]),
 tensor([[ 0.8484, -0.3442, -0.2634],
         [ 0.9591, -0.1435, -0.0263],
         [ 0.9458, -0.2209, -0.1128],
         [-0.8803,  0.2511,  0.1533],
         [-1.4574,  0.2959,  0.1316],
         [ 0.3941,  0.0685,  0.1370],
         [ 1.0905, -0.0797,  0.0675],
         [-0.7231,  0.2802,  0.2082]]))